# IEEE 33 barras: diagnóstico elétrico e capacidade de hospedagem de recarga VE

**Notebook científico reprodutível — benchmark inicial do VEOtimo**

Este notebook produz figuras e tabelas úteis para um manuscrito de alto impacto: topologia elétrica, perfil de tensão, perdas por trecho, carregamento espacial, cenários de recarga, curvas de estresse, capacidade de hospedagem e análise de sensibilidade locacional.

> **Escopo e cautela:** esta é a variante radial balanceada Baran–Wu com cargas PQ constantes. Ela não contém coordenadas urbanas, desequilíbrio trifásico, reguladores, proteção, incerteza nem rede viária. Portanto, os resultados validam o módulo elétrico e **não** representam uma cidade nem demonstram, isoladamente, uma localização ótima de eletropostos.

**Questões investigadas**

1. O fluxo reproduz os valores de referência do IEEE 33 barras?
2. Onde se concentram subtensão e perdas?
3. Como a mesma carga VE afeta barras eletricamente distintas?
4. Qual potência adicional cada barra admite antes de atingir o limite de tensão?
5. Como perdas e tensão respondem ao crescimento uniforme da demanda?


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import networkx as nx
import numpy as np
import pandas as pd

from veotimo.powerflow import load_network, run_power_flow

NETWORK_PATH = ROOT / "data" / "networks" / "ieee33.json"
FIGURES = ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {"base": "#164e63", "ev": "#dc2626", "limit": "#111827"}
V_MIN, V_MAX = 0.95, 1.05
network = load_network(NETWORK_PATH)
print(network["metadata"])


## 1. Auditoria do conjunto de dados

Antes do fluxo, verificam-se dimensão, radialidade, carga total e bases. A fonte e qualquer modificação da rede estão nos metadados do arquivo, evitando alterações silenciosas do benchmark.

In [ ]:
buses = pd.DataFrame(network["buses"]).set_index("id")
branches = pd.DataFrame(network["branches"])
summary = pd.Series({
    "barras": len(buses),
    "ramos_fechados": len(branches),
    "radial": len(branches) == len(buses) - 1,
    "carga_ativa_total_kW": buses.p_kw.sum(),
    "carga_reativa_total_kvar": buses.q_kvar.sum(),
    "tensao_base_kV": network["base"]["voltage_kv"],
    "potencia_base_MVA": network["base"]["power_mva"],
})
display(summary.to_frame("valor"))
assert summary["radial"] and summary["barras"] == 33
assert summary["carga_ativa_total_kW"] == 3715


## 2. Topologia do alimentador

O posicionamento abaixo é **esquemático**: o eixo horizontal representa profundidade elétrica e o vertical separa ramais. Não deve ser interpretado como geografia ou rede viária. Cor e tamanho representam a carga ativa local; a barra 1 é a subestação.

In [ ]:
G = nx.DiGraph()
G.add_nodes_from(buses.index)
G.add_edges_from(branches[["from", "to"]].itertuples(index=False, name=None))

def descendants_depth(graph, root=1):
    return nx.single_source_shortest_path_length(graph, root)

depth = descendants_depth(G)
# Posições editoriais determinísticas para separar tronco e ramais.
pos = {bus: (depth[bus], 0.0) for bus in range(1, 19)}
pos.update({bus: (depth[bus], 2.0) for bus in range(19, 23)})
pos.update({bus: (depth[bus], 1.0) for bus in range(23, 26)})
pos.update({bus: (depth[bus], -1.2) for bus in range(26, 34)})

fig, ax = plt.subplots(figsize=(16, 5.8))
loads = buses.p_kw.reindex(G.nodes).to_numpy()
nodes = nx.draw_networkx_nodes(G, pos, node_color=loads, node_size=90 + loads * 0.55,
                               cmap="viridis", edgecolors="white", linewidths=.8, ax=ax)
nx.draw_networkx_edges(G, pos, arrows=False, width=1.5, edge_color="#64748b", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, font_color="#111827", ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=[1], node_color="#ef4444", node_shape="s",
                       node_size=180, label="Subestação", ax=ax)
fig.colorbar(nodes, ax=ax, label="Carga ativa local (kW)", shrink=.78)
ax.set(title="Topologia esquemática do alimentador IEEE 33 barras",
       xlabel="Profundidade elétrica a partir da subestação", ylabel="Ramais (posição esquemática)")
ax.legend(loc="lower left"); ax.margins(.03); fig.tight_layout()
fig.savefig(FIGURES / "ieee33_topologia.png", dpi=300, bbox_inches="tight")
plt.show()


## 3. Caso base e validação numérica

A varredura backward/forward é um fluxo AC balanceado, não DC nem LinDistFlow. Os critérios de regressão adotados são tensão mínima de aproximadamente 0,9131 pu na barra 18 e perdas ativas de aproximadamente 202,68 kW.

In [ ]:
base = run_power_flow(network, voltage_min_pu=V_MIN, voltage_max_pu=V_MAX)
base_metrics = pd.Series({
    "convergiu": base.converged,
    "iteracoes": base.iterations,
    "barra_tensao_minima": base.min_voltage_bus,
    "tensao_minima_pu": base.bus_voltage_pu[base.min_voltage_bus],
    "perdas_ativas_kW": base.active_losses_kw,
    "perdas_reativas_kvar": base.reactive_losses_kvar,
    "perdas_percentual_carga": 100 * base.active_losses_kw / buses.p_kw.sum(),
    "barras_fora_limite": len(base.violations),
})
display(base_metrics.to_frame("valor"))
assert base.converged
assert np.isclose(base.bus_voltage_pu[18], .9131, atol=1e-4)
assert np.isclose(base.active_losses_kw, 202.68, atol=.1)


In [ ]:
voltage = pd.Series(base.bus_voltage_pu, name="Tensão (pu)").sort_index()
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.plot(voltage.index, voltage, marker="o", ms=4, lw=2, color=COLORS["base"], label="Caso base")
ax.axhspan(0, V_MIN, color="#fecaca", alpha=.45, label="Região de subtensão")
ax.axhline(V_MIN, color=COLORS["limit"], ls="--", lw=1.2, label=f"Limite {V_MIN:.2f} pu")
ax.scatter([base.min_voltage_bus], [voltage.loc[base.min_voltage_bus]], color="#dc2626", zorder=5)
ax.annotate(f"mín. {voltage.min():.4f} pu", (base.min_voltage_bus, voltage.min()),
            xytext=(-70, 25), textcoords="offset points", arrowprops={"arrowstyle": "->"})
ax.set(xlabel="Barra", ylabel="Magnitude de tensão (pu)", title="Perfil de tensão — caso base", xticks=range(1,34,2))
ax.legend(ncol=3); fig.tight_layout()
fig.savefig(FIGURES / "ieee33_perfil_tensao_base.png", dpi=300, bbox_inches="tight")
plt.show()


## 4. Decomposição espacial das perdas

As perdas por ramo são calculadas por $P^{loss}_{ij}=|I_{ij}|^2R_{ij}S_{base}$. Isso permite identificar trechos prioritários para análise de reforço. Não equivale a uma decisão automática de investimento.

In [ ]:
s_base_kva = network["base"]["power_mva"] * 1000
z_base = network["base"]["voltage_kv"]**2 / network["base"]["power_mva"]
loss_rows = []
for row in branches.itertuples(index=False):
    edge = (row[0], row[1])
    current = base.branch_current_pu[edge]
    loss_rows.append({"ramo": f"{edge[0]}–{edge[1]}", "de": edge[0], "para": edge[1],
                      "corrente_pu": current,
                      "perda_ativa_kW": current**2 * (row.r_ohm / z_base) * s_base_kva})
losses = pd.DataFrame(loss_rows).sort_values("perda_ativa_kW", ascending=False)
display(losses.head(10).style.format({"corrente_pu": "{:.5f}", "perda_ativa_kW": "{:.2f}"}))

fig, ax = plt.subplots(figsize=(10, 5.5))
top = losses.head(12).sort_values("perda_ativa_kW")
ax.barh(top.ramo, top.perda_ativa_kW, color="#0f766e")
ax.set(xlabel="Perda ativa (kW)", ylabel="Ramo", title="Doze ramos com maiores perdas — caso base")
fig.tight_layout(); fig.savefig(FIGURES / "ieee33_perdas_por_ramo.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Efeito locacional de uma estação de recarga

Compara-se a **mesma** estação de 120 kW e fator de potência 0,95 em barras com diferentes distâncias elétricas. Esta experiência isola o efeito elétrico da localização; acessibilidade e demanda viária ainda não estão incluídas.

In [ ]:
station_kw, station_pf = 120.0, .95
station_kvar = station_kw * np.tan(np.arccos(station_pf))
candidate_buses = [6, 18, 25, 33]
scenario_results = []
profiles = {"Base": voltage}
for bus in candidate_buses:
    result = run_power_flow(network, {bus: (station_kw, station_kvar)}, voltage_min_pu=V_MIN)
    profiles[f"VE @ barra {bus}"] = pd.Series(result.bus_voltage_pu).sort_index()
    scenario_results.append({
        "barra_conexao": bus,
        "potencia_kW": station_kw,
        "tensao_minima_pu": min(result.bus_voltage_pu.values()),
        "barra_tensao_minima": result.min_voltage_bus,
        "perdas_ativas_kW": result.active_losses_kw,
        "incremento_perdas_kW": result.active_losses_kw - base.active_losses_kw,
        "barras_fora_limite": len(result.violations),
    })
comparison = pd.DataFrame(scenario_results).set_index("barra_conexao")
display(comparison.style.format(precision=3).background_gradient(subset=["incremento_perdas_kW"], cmap="Reds"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for label, profile in profiles.items():
    axes[0].plot(profile.index, profile.values, lw=1.6, label=label)
axes[0].axhline(V_MIN, color="black", ls="--", lw=1)
axes[0].set(xlabel="Barra", ylabel="Tensão (pu)", title="Perfis de tensão por ponto de conexão")
axes[0].legend(fontsize=8, ncol=2)
comparison.incremento_perdas_kW.plot.bar(ax=axes[1], color="#b91c1c")
axes[1].set(xlabel="Barra de conexão", ylabel="Incremento de perdas (kW)", title="Penalidade elétrica locacional")
fig.tight_layout(); fig.savefig(FIGURES / "ieee33_cenarios_localizacao_ve.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Curva de estresse da demanda

Escala-se toda a carga mantendo o fator de potência. A curva evidencia não linearidade das perdas e margem de tensão. É uma sensibilidade determinística, não uma previsão de crescimento.

In [ ]:
multipliers = np.linspace(.5, 1.5, 21)
stress = []
for factor in multipliers:
    additions = {bus: ((factor-1)*row.p_kw, (factor-1)*row.q_kvar) for bus, row in buses.iterrows()}
    result = run_power_flow(network, additions, voltage_min_pu=V_MIN)
    stress.append({"fator_carga": factor, "tensao_minima_pu": min(result.bus_voltage_pu.values()),
                   "perdas_ativas_kW": result.active_losses_kw, "convergiu": result.converged})
stress = pd.DataFrame(stress)
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(stress.fator_carga, stress.tensao_minima_pu, "o-", color="#1d4ed8", label="Tensão mínima")
ax2.plot(stress.fator_carga, stress.perdas_ativas_kW, "s-", color="#b91c1c", label="Perdas")
ax1.axhline(V_MIN, color="black", ls="--", lw=1)
ax1.set(xlabel="Multiplicador uniforme da carga", ylabel="Tensão mínima (pu)", title="Curva de estresse elétrico")
ax2.set_ylabel("Perdas ativas (kW)")
lines = ax1.lines[:1] + ax2.lines
ax1.legend(lines, [line.get_label() for line in lines], loc="center left")
fig.tight_layout(); fig.savefig(FIGURES / "ieee33_curva_estresse.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Capacidade de hospedagem locacional

Para cada barra de carga, uma busca binária encontra a maior potência ativa VE adicional que mantém $V_{min}\geq0{,}95$ pu, com fator de potência 0,95 e teto de busca de 2 MW. Se o caso base já viola o limite — como ocorre neste benchmark — a capacidade incremental estrita é zero. Para ainda estudar a **margem incremental**, também se calcula o acréscimo permitido antes de uma queda adicional configurável de 0,01 pu.

Essa distinção evita apresentar como “capacidade disponível” uma rede cujo caso base já é tecnicamente inadequado.

In [ ]:
def hosting_capacity(bus, threshold, pf=.95, upper_kw=2000., iterations=30):
    low, high = 0., upper_kw
    q_ratio = np.tan(np.arccos(pf))
    if min(run_power_flow(network).bus_voltage_pu.values()) < threshold:
        return 0.0
    for _ in range(iterations):
        mid = (low + high) / 2
        result = run_power_flow(network, {bus: (mid, mid*q_ratio)}, voltage_min_pu=threshold)
        if result.converged and min(result.bus_voltage_pu.values()) >= threshold:
            low = mid
        else:
            high = mid
    return low

base_vmin = min(base.bus_voltage_pu.values())
strict = {bus: hosting_capacity(bus, V_MIN) for bus in buses.index if bus != 1}
incremental_threshold = base_vmin - .01
incremental = {bus: hosting_capacity(bus, incremental_threshold) for bus in buses.index if bus != 1}
hosting = pd.DataFrame({"estrita_0_95_kW": strict, "margem_queda_0_01pu_kW": incremental})
display(hosting.sort_values("margem_queda_0_01pu_kW", ascending=False).head(10).style.format("{:.1f}"))

fig, ax = plt.subplots(figsize=(13, 4.8))
ax.bar(hosting.index, hosting.margem_queda_0_01pu_kW, color="#7c3aed")
ax.set(xlabel="Barra de conexão", ylabel="Potência VE adicional (kW)",
       title=f"Margem incremental até Vmin = {incremental_threshold:.4f} pu (queda adicional de 0,01 pu)", xticks=range(2,34,2))
fig.tight_layout(); fig.savefig(FIGURES / "ieee33_capacidade_hospedagem.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. Síntese espacial multicritério elétrica

O painel combina tensão, carga e margem incremental. Ele é um **diagnóstico elétrico**, não um ranking final de eletropostos. Um artigo do método integrado deverá acrescentar demanda OD, distância/tempo em rede viária, fila, custos, transformadores e incerteza; somente então comparar as estratégias mobilidade, elétrica e integrada.

In [ ]:
diagnostic = buses.copy()
diagnostic["tensao_base_pu"] = voltage
diagnostic["profundidade"] = pd.Series(depth)
diagnostic["margem_incremental_ve_kW"] = pd.Series(incremental)
diagnostic["subtensao"] = diagnostic.tensao_base_pu < V_MIN

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0,0].scatter(diagnostic.profundidade, diagnostic.tensao_base_pu,
                  s=40+diagnostic.p_kw/3, c=diagnostic.p_kw, cmap="viridis", alpha=.85)
axes[0,0].axhline(V_MIN, color="black", ls="--"); axes[0,0].set(xlabel="Profundidade elétrica", ylabel="Tensão (pu)", title="Tensão × profundidade")
axes[0,1].scatter(diagnostic.p_kw, diagnostic.tensao_base_pu, c=diagnostic.profundidade, cmap="plasma", s=55)
axes[0,1].set(xlabel="Carga local (kW)", ylabel="Tensão (pu)", title="Carga local × tensão")
axes[1,0].hist(voltage, bins=10, color="#0369a1", edgecolor="white"); axes[1,0].axvline(V_MIN, color="black", ls="--")
axes[1,0].set(xlabel="Tensão (pu)", ylabel="Número de barras", title="Distribuição das tensões")
axes[1,1].scatter(diagnostic.tensao_base_pu, diagnostic.margem_incremental_ve_kW,
                  c=diagnostic.profundidade, cmap="plasma", s=55)
axes[1,1].set(xlabel="Tensão base (pu)", ylabel="Margem incremental VE (kW)", title="Tensão × margem incremental")
fig.suptitle("Painel de diagnóstico elétrico IEEE 33 barras", fontsize=15)
fig.tight_layout(); fig.savefig(FIGURES / "ieee33_painel_diagnostico.png", dpi=300, bbox_inches="tight")
plt.show()

diagnostic.to_csv(ROOT / "results" / "ieee33_diagnostico_barras.csv", index_label="bus_id")
comparison.to_csv(ROOT / "results" / "ieee33_cenarios_localizacao.csv")
stress.to_csv(ROOT / "results" / "ieee33_curva_estresse.csv", index=False)
hosting.to_csv(ROOT / "results" / "ieee33_capacidade_hospedagem.csv", index_label="bus_id")
print("Tabelas e figuras exportadas para", ROOT / "results")


## 9. Interpretação científica e próximos experimentos

### Achados que podem ser reportados após execução

- reprodutibilidade do benchmark pelos indicadores de referência;
- distribuição espacial das violações e concentração das perdas;
- diferença do impacto elétrico da mesma potência conforme a barra de conexão;
- resposta não linear das perdas ao crescimento da carga;
- distinção entre limite absoluto e margem incremental de hospedagem.

### O que **não** pode ser concluído deste notebook

- que uma barra elétrica seja a melhor localização urbana;
- que a demanda VE seja 120 kW ou possua simultaneidade unitária;
- que a rede represente uma distribuidora/cidade;
- que o modelo balanceado reproduza uma rede trifásica real;
- que correlação visual implique causalidade ou significância estatística.

### Extensões recomendadas para artigo A1

1. integrar rede viária sintética e, depois, real, mantendo topologias elétrica e urbana distintas;
2. gerar recarga a partir de viagens OD, SOC e energia, com perfis de 15 minutos;
3. executar Monte Carlo para demanda, disponibilidade, preço e crescimento;
4. validar soluções candidatas em pandapower/OpenDSS trifásico, incluindo reguladores e limites térmicos;
5. comparar mobilidade-only, electrical-only e integrated com mesmas sementes;
6. comparar MILP em instâncias pequenas a GA/PSO/GWO/NSGA-II em escala;
7. reportar Pareto, hipervolume, factibilidade, tempo, avaliações e testes estatísticos;
8. realizar análise de sensibilidade global e validação fora da amostra;
9. publicar configuração, dados permitidos, versões e manifesto por execução.

As figuras são salvas em 300 dpi e as tabelas em CSV, permitindo revisão independente e composição posterior do manuscrito.
